# B2.4 · Budgets and stop conditions

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *AI for Security*

Builds on **[B2.3 · Tool design](https://spbreed.github.io/cyber-commons/lessons/B2.3.html)**.

| | |
|---|---|
| Open-source tooling | Python |
| Open-weight models | Llama 3.3 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Budgets are the only control that still works when everything else has failed —
including the verifier.

Two budgets, bounding different things:

- **Step budget** bounds *cost*. It caps how many times the loop can iterate.
- **Time budget** bounds *damage*. A loop calling a fast tool can complete
  thousands of actions in the seconds a step budget still permits.

You need both, and you need to decide what happens when one fires. There are
three options and they are not equivalent:

1. **Halt** — stop and leave the world as it is. Safe when actions are additive.
2. **Roll back** — undo the partial work. Requires every action to be undoable.
3. **Escalate** — hand to a human with the trace. The only right answer when
   the work is half-done and not undoable.

A harness that halts mid-way through a multi-step change and cannot say which
steps completed has not been contained; it has been abandoned.

## 2 · Demo — a loop that never converges

In [ ]:
import time
from dataclasses import dataclass, field

class ReplayModel:
    """DETERMINISTIC REPLAY — not a language model."""
    def __init__(self, proposals): self.proposals, self.calls = list(proposals), 0
    def propose(self, _):
        p = self.proposals[min(self.calls, len(self.proposals)-1)]; self.calls += 1
        return p

def never_satisfied(_): return False, "still not right"

def run(model, verifier, max_steps=5, max_seconds=10.0):
    steps, started = [], time.monotonic()
    for n in range(1, max_steps+1):
        p = model.propose("")
        ok, why = verifier(p)
        steps.append((n, p, ok, why))
        if ok:   return steps, "verifier satisfied"
        if time.monotonic() - started > max_seconds:
            return steps, f"time budget ({max_seconds}s)"
    return steps, f"step budget ({max_steps} steps)"

spinner = lambda: ReplayModel(["retrying the same approach"])
for limit in (1, 3, 10):
    steps, why = run(spinner(), never_satisfied, max_steps=limit)
    print(f"max_steps={limit:>3} → {len(steps):>3} steps, stopped by {why}")

steps, why = run(spinner(), never_satisfied, max_steps=10_000, max_seconds=0.05)
print(f"\ntime budget      → {len(steps):>3} steps, stopped by {why}")
print("The step budget alone would have permitted 10,000 iterations.")

## 3 · Where it breaks — the loop stops half-way through a change

Budgets that only halt leave the system in a state nobody chose. Here is a three-step remediation that gets cut off after step two.

In [ ]:
@dataclass
class World:
    firewall_rule_added: bool = False
    service_restarted: bool = False
    monitoring_updated: bool = False
    def state(self):
        return {k: v for k, v in vars(self).items()}

PLAN = [("add firewall rule",  "firewall_rule_added"),
        ("restart service",    "service_restarted"),
        ("update monitoring",  "monitoring_updated")]

def apply_plan(world, budget):
    done = []
    for i, (label, attr) in enumerate(PLAN, 1):
        if i > budget:
            return done, f"budget exhausted after step {i-1}"
        setattr(world, attr, True)
        done.append(label)
    return done, "complete"

w = World()
done, why = apply_plan(w, budget=2)
print("plan:", [p[0] for p in PLAN])
print("done:", done)
print("why :", why)
print("world state:", w.state())
print("\nThe firewall now blocks traffic the service needs, the service has been")
print("restarted into that condition, and monitoring does not know to alert.")
print("Halting was worse than either finishing or never starting.")

## 4 · The control — choose the stop behaviour per action class

In [ ]:
ACTIONS = {
 # action                 undoable?  safe to leave half-done?
 "add firewall rule":     (True,     False),
 "restart service":       (False,    False),
 "update monitoring":     (True,     True),
 "post a comment":        (False,    True),
 "open a pull request":   (True,     True),
 "merge a pull request":  (False,    False),
}
def stop_behaviour(action):
    undoable, safe_partial = ACTIONS[action]
    if safe_partial:            return "HALT — additive, safe to leave"
    if undoable:                return "ROLL BACK — undo what completed"
    return "ESCALATE — half-done and not undoable; hand the trace to a human"

for a in ACTIONS:
    print(f"{a:24s}{stop_behaviour(a)}")

In [ ]:
# Verify: a budgeted run that rolls back or escalates correctly.
@dataclass
class Runner:
    world: World = field(default_factory=World)
    applied: list = field(default_factory=list)

    def apply(self, label, attr):
        setattr(self.world, attr, True); self.applied.append((label, attr))

    def rollback(self):
        undone = []
        for label, attr in reversed(self.applied):
            if ACTIONS[label][0]:                       # undoable
                setattr(self.world, attr, False); undone.append(label)
            else:
                return undone, f"cannot undo {label!r} — escalating"
        return undone, "fully rolled back"

    def run(self, plan, budget):
        for i, (label, attr) in enumerate(plan, 1):
            if i > budget:
                worst = [l for l, _ in self.applied if not ACTIONS[l][1]]
                if not worst:
                    return "HALT", "all completed actions are safe to leave"
                undone, detail = self.rollback()
                return ("ESCALATE" if "escalat" in detail else "ROLLBACK"), detail
            self.apply(label, attr)
        return "COMPLETE", "plan finished"

for budget in (1, 2, 3):
    r = Runner()
    verdict, detail = r.run(PLAN, budget)
    print(f"budget={budget}  {verdict:9s} {detail}")
    print(f"          world: {r.world.state()}")

## What you just proved

Each step budget is honoured exactly; the time budget stops the loop far short of 10,000 steps. The half-applied plan leaves a firewall rule blocking a service that was then restarted. The per-action table assigns HALT, ROLL BACK and ESCALATE correctly, and the budgeted runner rolls back at budget 1 and escalates at budget 2 because the service restart cannot be undone.

## Your turn

Classify every action your harness can take into the three columns. The ones that are neither undoable nor safe to leave half-done are the ones that need a human in the escalation path — and they are usually the ones nobody has thought about.

---

**Next → [B2.5 · Model tiering and routing inside the loop](https://spbreed.github.io/cyber-commons/lessons/B2.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*